# Descargar las estaciones reales de Costa Rica que faltan (ECO | Wind)

Pablo notó que la muestra de 4 sitios precacheados (San José, Nicoya, Liberia, Finca Favorita) es
pobre para un territorio tan accidentado como Costa Rica -- Hallazgo 21/22/23 ya mostraron que la
forma real del viento cambia mucho de una zona a otra. El catálogo completo
(`datos_clima/epw_catalog_global.json`, 5,276 estaciones/20 países, Hallazgo 19) en realidad sólo
tiene **12 estaciones para Costa Rica en total** -- ya tenemos 4, faltan **8**. Este notebook las
descarga todas, igual que ya lo hace la app (`descargar_y_extraer_epw()`, mismo patrón que
DDP-lite/Skyplus) -- no hay que ir sitio por sitio a mano.

**No corre en el sandbox de desarrollo** (climate.onebuilding.org bloqueado, Hallazgo 2) -- correr
esto en Colab. Al final arma un .zip con los 8 EPW nuevos para descargar y subir de vuelta al chat
(mismo mecanismo que ya se usó para los primeros 3 EPW reales, Hallazgo 18).

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 3), reused 2 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.44 KiB | 379.00 KiB/s, done.


From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD
   4566c9e..3e656d6  main       -> origin/main


HEAD is now at 3e656d6 Merge pull request #2 from Sogo2012/claude/refactor-ui-tabs-layout-hmc5wt


/home/user/eco-wind/notebooks
Commit activo: 3e656d6  Merge pull request #2 from Sogo2012/claude/refactor-ui-tabs-layout-hmc5wt  (2026-08-31 15:12:00 -0600)


In [2]:
import sys
sys.path.insert(0, "..")

import json
import shutil
import zipfile

from engine.epw_real import (
    CARPETA_EPW_REAL, _SITIOS_PRECACHEADOS_COORDS, cargar_epw_real,
    descargar_y_extraer_epw, _haversine_km,
)

catalogo = json.load(open(os.path.join(repo, "datos_clima", "epw_catalog_global.json")))
cri = catalogo["CRI"]
print(f"Catálogo de Costa Rica: {len(cri)} estaciones en total.")

# Las 4 que ya tenemos, identificadas por nombre (son las únicas 4 de las 12 que ya están
# precacheadas -- ver _SITIOS_PRECACHEADOS_COORDS en engine/epw_real.py).
YA_TENEMOS = {"San Jose Santamaria Intl AP", "Nicoya AP", "Quiros Liberia Intl AP", "Finca Favorita"}
faltantes = [s for s in cri if s["name"] not in YA_TENEMOS]

print(f"Ya tenemos {len(YA_TENEMOS)} localmente. Faltan {len(faltantes)}:")
for s in faltantes:
    print(f"  - {s['name']} ({s.get('state', '?')})")

Catálogo de Costa Rica: 12 estaciones en total.
Ya tenemos 4 localmente. Faltan 8:
  - Limon Intl AP (LI)
  - Chacarita Puntarenas AP (PU)
  - Palmar Sur Southern Zone Intl AP (PU)
  - Parrita (PU)
  - Paso Canoas AP (PU)
  - Puntarenas (PU)
  - San Jose Bolanos Intl AP (SJ)
  - San Jose La Sabana (SJ)


## Descargar y verificar cada una

Para cada estación faltante: descarga el ZIP real de climate.onebuilding.org, extrae el .epw, lo
copia a `datos_clima/epw_real/` (misma carpeta que las otras 3), y lo vuelve a abrir con
`cargar_epw_real()` para confirmar que se lee bien -- reportando la media real de viento, la
elevación y la coordenada real que trae el propio encabezado del archivo (más confiable que la del
catálogo: la de Finca Favorita, por ejemplo, ya se sabía que no coincidía -- ver el comentario
arriba de `_SITIOS_PRECACHEADOS_COORDS` en `engine/epw_real.py`).

In [3]:
os.makedirs(CARPETA_EPW_REAL, exist_ok=True)
descargadas = []

for s in faltantes:
    print(f"=== {s['name']} ===")
    try:
        ruta_tmp = descargar_y_extraer_epw(s["url"])
        ruta_final = os.path.join(CARPETA_EPW_REAL, os.path.basename(ruta_tmp))
        shutil.copy(ruta_tmp, ruta_final)

        df, meta = cargar_epw_real(ruta_final)
        print(f"  OK -- {os.path.basename(ruta_final)}")
        print(f"  Media real: {df['WS10M'].mean():.3f} m/s | elevación: {meta['elevacion_m']:.0f} m | "
              f"lat={meta['lat']:.4f}, lon={meta['lon']:.4f}")

        # Distancia al sitio precacheado real más cercano de los 4 que ya teníamos --
        # da una primera idea de si esto aporta cobertura nueva o es redundante.
        dist_min = min(_haversine_km(meta["lat"], meta["lon"], slat, slon)
                        for slat, slon in _SITIOS_PRECACHEADOS_COORDS.values())
        print(f"  Distancia al más cercano de los 4 ya conocidos: {dist_min:.1f} km")

        descargadas.append(dict(nombre=s["name"], archivo=os.path.basename(ruta_final),
                                 media_m_s=float(df["WS10M"].mean()), elevacion_m=meta["elevacion_m"],
                                 lat=meta["lat"], lon=meta["lon"], dist_km_mas_cercano=dist_min))
    except Exception as exc:
        print(f"  FALLO: {exc!r}")
    print()

print(f"Descargadas con éxito: {len(descargadas)}/{len(faltantes)}")

=== Limon Intl AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_LI_Limon.Intl.AP.787670_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Chacarita Puntarenas AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Chacarita-Puntarenas.AP.787613_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Palmar Sur Southern Zone Intl AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Palmar.Sur-Southern.Zone.Intl.AP.787720_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Parrita ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Parrita.749036_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Paso Canoas AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Paso.Canoas.AP.787606_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Puntarenas ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_PU_Puntarenas.787600_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== San Jose Bolanos Intl AP ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_SJ_San.Jose-Bolanos.Intl.AP.787640_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== San Jose La Sabana ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/CRI_Costa_Rica/CRI_SJ_San.Jose-La.Sabana.787605_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

Descargadas con éxito: 0/8


## Empaquetar para subir de vuelta al chat

Un solo .zip con los EPW nuevos + un resumen en JSON -- descargalo del navegador de archivos de
Colab (ícono de carpeta a la izquierda) y subilo acá en el chat. Con eso integro los sitios nuevos
en `engine/epw_real.py`, corro de nuevo Hallazgo 21-23 con la muestra ampliada, documento y hago
commit+push -- igual que con los primeros 3 EPW (Hallazgo 18).

In [4]:
ruta_zip = os.path.join(repo, "estaciones_cr_nuevas.zip")
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for d in descargadas:
        z.write(os.path.join(CARPETA_EPW_REAL, d["archivo"]), arcname=d["archivo"])
    z.writestr("resumen.json", json.dumps(descargadas, indent=2, ensure_ascii=False))

print(f"Armado: {ruta_zip} ({os.path.getsize(ruta_zip) / 1024:.0f} KB, {len(descargadas)} estaciones)")

try:
    from google.colab import files
    files.download(ruta_zip)
except ImportError:
    print("No estás en Colab -- el archivo quedó en el disco local, en la ruta de arriba.")

Armado: /home/user/eco-wind/estaciones_cr_nuevas.zip (0 KB, 0 estaciones)
No estás en Colab -- el archivo quedó en el disco local, en la ruta de arriba.
